# 04 - PLS variants

**Purpose**: PLSCanonical fits between connectivity (X-block) and
kinematics (Y-block) for each of three research questions:

- ``injury_snapshot`` -- kinematic profile at Post_Injury_2-4.
- ``deficit_delta``  -- change from Baseline to Post_Injury_2-4.
- ``recovery_delta`` -- change from Post_Injury_2-4 to Post_Rehab_Test.

Set ``VARIANT`` in the parameters cell to pick which one to run, or
leave it as ``'all'`` to run them sequentially.

**Input**: ``data.FCDGdf_wide`` and ``data.AKDdf_agg_contact_flat()``
restricted to the important regions / features identified in 01 and 02.


In [ ]:
# parameters
VARIANT = 'all'  # 'injury_snapshot', 'deficit_delta', 'recovery_delta', or 'all'
N_COMPONENTS = 2
TOP_N = 15       # Top connectivity regions to label per LV


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from endpoint_ck_analysis.config import CACHE_DIR, EXAMPLE_OUTPUT_DIR
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.dimreduce import build_y_phase, build_y_shift, run_pls
from endpoint_ck_analysis.helpers.plotting import plot_pls, stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data = load_all()
agg_flat = data.AKDdf_agg_contact_flat()

# Pull the important regions / features written by notebooks 01 and 02.
important_regions = pd.read_parquet(CACHE_DIR / 'important_regions.parquet')['region_hemi'].tolist()
important_features = pd.read_parquet(CACHE_DIR / 'important_features.parquet')['feature'].tolist()

X_block = data.FCDGdf_wide[important_regions].fillna(0)


## 1. Build the three Y-blocks

These share the same X-block (connectivity) so results compare directly.


In [ ]:
Y_injury = build_y_phase(agg_flat, important_features, 'Post_Injury_2-4')
Y_deficit = build_y_shift(agg_flat, important_features, 'Baseline', 'Post_Injury_2-4')
Y_recovery = build_y_shift(agg_flat, important_features, 'Post_Injury_2-4', 'Post_Rehab_Test')

Y_BLOCKS = {
    'injury_snapshot': (Y_injury, 'Injury snapshot (Post_Injury_2-4 kinematics)'),
    'deficit_delta':   (Y_deficit, 'Injury deficit (Post_Injury_2-4 - Baseline)'),
    'recovery_delta':  (Y_recovery, 'ABT recovery (Post_Rehab_Test - Post_Injury_2-4)'),
}


## 2. Run the selected variant(s)

If ``VARIANT == 'all'`` every variant runs in order; otherwise only the
selected one. Each call produces a three-panel figure per LV
(X-loadings, Y-loadings, score-vs-score scatter).


In [ ]:
variants = list(Y_BLOCKS.keys()) if VARIANT == 'all' else [VARIANT]
if VARIANT != 'all' and VARIANT not in Y_BLOCKS:
    raise ValueError(f"Unknown VARIANT {VARIANT!r}. Choose one of {list(Y_BLOCKS) + ['all']}.")

results = {}
for variant in variants:
    Y, label = Y_BLOCKS[variant]
    print(f'\n=== {variant} ===')
    results[variant] = run_pls(X_block, Y, n_components=N_COMPONENTS, label=label)
    plot_pls(results[variant], top_n=TOP_N)


## 3. Export latent-variable scores for the gallery

Each variant's subject scores are written to the cache so the figure
gallery can assemble a summary figure without re-fitting.


In [ ]:
for variant, r in results.items():
    out = pd.DataFrame(r['X_scores'], index=r['subjects'],
                       columns=[f'LV{i+1}' for i in range(r['X_scores'].shape[1])])
    out.to_parquet(CACHE_DIR / f'pls_{variant}_X_scores.parquet')
    print(f'Wrote pls_{variant}_X_scores.parquet, N={len(out)}')


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the three PLS results tell you (click to expand)</summary>

**Each variant produces 3 panels per latent variable**: connectivity
loadings (X), kinematic loadings (Y), subject cross-score scatter.

**Cross-score scatter** is the headline.

- All subjects falling on or near the diagonal with high r (>0.8) =
  connectivity X-block and kinematic Y-block share a real coupled
  axis. PLS found a connectivity pattern that tracks a kinematic
  pattern across the cohort.
- Spread cloud, low r = no coherent axis was found.
- At small N, **every PLS will look highly correlated by construction**.
  PLSCanonical is given enough freedom (4 subjects x dozens of features
  per side) that it can always find a fit. Treat the cross-score r as
  descriptive ("here's what the data shows"), NOT inferential ("there's
  a real relationship"). At higher N this becomes a real test.

**Variant interpretations**:

- *Injury snapshot*: which connectivity regions covary with which
  kinematic features at the post-injury timepoint. Heads up the "this
  injury produces this kinematic deficit" story.
- *Deficit delta*: which connectivity regions track HOW FAR a subject
  fell from baseline. The X-loadings here are the regions whose sparing
  buffers vs amplifies the immediate deficit.
- *Recovery delta*: which connectivity regions track HOW MUCH a subject
  recovered with ABT. The X-loadings here are the regions whose
  sparing predicts rehab response.

**Loadings interpretation** (same for all three variants): high
absolute loading on LV1 means the region/feature contributes most to
the dominant coupled axis. Sign tells you the direction (positive
loading on X side + positive on Y side = these covary; opposite signs
= they covary inversely).

</details>
